# 02b · Phase 2 gate — Are the clean tables consistent enough to build the database?

Reads **only** `data/clean/*.csv` (as a later loader would) and checks them *before* any Django model exists.
It also re-counts rows and NULLs **independently from the raw JSON** (different code path), so a bug in `cleaning.py` cannot hide itself.

| Family | Question |
|---|---|
| schema | right columns, types, NULLs only where allowed, no blank/padded strings? |
| keys | primary keys unique? every foreign key points to an existing row? |
| domain | enums, patterns (CVE-…, T####), ranges (CVSS 0–10), date bounds? |
| semantic | rules across columns (score ↔ severity, flag ↔ dates, parent ↔ id)? |
| reconcile | row and NULL counts == independent recount from raw? |
| provenance | files == clean manifest, lossless round trip, raw still untouched? |

In [1]:
import sys, json, shutil, tempfile
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
from IPython.display import display
from threatgraph import config, cleaning as C, validate_clean as V
from threatgraph.manifest import read_manifest
from threatgraph.integrity import sha256_file, IntegrityError
pd.set_option("display.max_columns", 30, "display.max_colwidth", 70, "display.width", 200, "display.max_rows", 100)
SRC, RAW, CLEAN = config.load_sources(), config.RAW_DIR, config.CLEAN_DIR
raw_manifest, clean_manifest = read_manifest(config.MANIFEST_PATH), json.loads((CLEAN / "manifest.json").read_text())

## 1 · Load the clean tables from disk (typed)

In [2]:
tables = {name: C.read_clean(name, CLEAN) for name in C.SCHEMA}
display(pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}, "columns": {k: v.shape[1] for k, v in tables.items()}}))
display(tables["nvd_cve"].drop(columns=["description_en"]).head(3))
display(tables["pkev_mapping"].head(3))

,rows,columns
kev_vulnerability,1391,12
kev_cwe,1328,2
nvd_cve,1391,13
nvd_cve_cwe,2523,5
attack_tactic,14,4
attack_technique,799,9
attack_technique_tactic,1051,2
pkev_mapping,1183,7
pkev_mapping_reference,2256,3


,cve_id,source_identifier,published,last_modified,vuln_status,cvss_v31_score,cvss_v31_severity,cvss_v31_vector,cvss_v31_source,cvss_v31_type,cvss_v2_score,n_references
0,CVE-2002-0367,cve@mitre.org,2002-06-25 04:00:00+00:00,2025-04-03 01:03:51.193000+00:00,Deferred,7.8,HIGH,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H,nvd@nist.gov,Primary,7.2,18
1,CVE-2004-0210,cve@mitre.org,2004-08-06 04:00:00+00:00,2025-04-03 01:03:51.193000+00:00,Deferred,7.8,HIGH,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H,nvd@nist.gov,Primary,7.2,12
2,CVE-2004-1464,cve@mitre.org,2004-12-31 05:00:00+00:00,2025-04-03 01:03:51.193000+00:00,Deferred,5.9,MEDIUM,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:N/I:N/A:H,nvd@nist.gov,Primary,5.0,12


,mapping_id,cve_id,technique_id,mapping_type,capability_group,comments,status
0,1,CVE-2024-34102,T1190,exploitation_technique,xxe,This vulnerability is exploited by sending a crafted XML document ...,complete
1,10,CVE-2024-4671,T1189,exploitation_technique,use_after_free,CVE-2024-4671 is a use-after-free vulnerability where an adversary...,complete
2,100,CVE-2023-48788,T1059,primary_impact,sql_injection,This is an SQL injection vulnerability that can be exploited to ex...,complete


## 2 · The six families of checks

In [3]:
raw = C.load_raw(RAW)                                   # read-only, only for the independent recount and the cross-source report
expected = V.expected_from_raw(raw)
results = (V.check_schema(tables) + V.check_keys(tables) + V.check_domain(tables, SRC) + V.check_semantic(tables)
           + V.check_reconcile(tables, expected) + V.check_provenance(CLEAN, clean_manifest, RAW, raw_manifest, tables))
summary = pd.DataFrame(V.summarize(results)).T; summary["total"] = summary.passed + summary.failed
display(summary)
print("FAILED:", [r for r in results if not r.ok] or "none")

,passed,failed,total
schema,45,0,45
keys,22,0,22
domain,28,0,28
semantic,8,0,8
reconcile,19,0,19
provenance,30,0,30


FAILED: none


In [4]:
# Nothing hidden behind a green total: the referential-integrity checks one by one.
pd.DataFrame([r.__dict__ for r in results if r.family == "keys"])[["name", "ok", "detail"]]

,name,ok,detail
0,kev_vulnerability: primary key ['cve_id'] unique,True,0 duplicates
1,"kev_cwe: primary key ['cve_id', 'cwe_id'] unique",True,0 duplicates
2,kev_cwe['cve_id'] -> kev_vulnerability['cve_id'],True,0 orphans: []
3,nvd_cve: primary key ['cve_id'] unique,True,0 duplicates
4,nvd_cve['cve_id'] -> kev_vulnerability['cve_id'],True,0 orphans: []
5,"nvd_cve_cwe: primary key ['cve_id', 'cwe_id', 'source'] unique",True,0 duplicates
6,nvd_cve_cwe['cve_id'] -> nvd_cve['cve_id'],True,0 orphans: []
7,attack_tactic: primary key ['tactic_id'] unique,True,0 duplicates
8,attack_tactic: unique ['shortname'],True,0 duplicates
9,attack_tactic: unique ['stix_id'],True,0 duplicates


## 3 · Reconciliation, shown (clean vs. independent recount from raw)

In [5]:
rec = pd.DataFrame([{"check": r.name, "ok": r.ok, "detail": r.detail} for r in results if r.family == "reconcile"])
display(rec)

,check,ok,detail
0,kev_vulnerability: rows == independent recount,True,clean 1391 vs raw 1391
1,kev_cwe: rows == independent recount,True,clean 1328 vs raw 1328
2,nvd_cve: rows == independent recount,True,clean 1391 vs raw 1391
3,nvd_cve_cwe: rows == independent recount,True,clean 2523 vs raw 2523
4,attack_tactic: rows == independent recount,True,clean 14 vs raw 14
5,attack_technique: rows == independent recount,True,clean 799 vs raw 799
6,attack_technique_tactic: rows == independent recount,True,clean 1051 vs raw 1051
7,pkev_mapping: rows == independent recount,True,clean 1183 vs raw 1183
8,pkev_mapping_reference: rows == independent recount,True,clean 2256 vs raw 2256
9,nvd_cve.cvss_v31_score == independent recount,True,clean 2 vs raw 2


## 4 · Pinned numbers
The cleaning-rule counters and the cross-source disagreements are **measured once** and pinned in `config/sources.yaml` (`expected_clean`).
If one changes, either the data or the rules changed and we must look.

In [6]:
exp = SRC["expected_clean"]
rows_ok = {t: len(tables[t]) == n for t, n in exp["rows"].items()}
rules_ok = {k: clean_manifest["rule_counts"].get(k) == v for k, v in exp["rule_counts"].items()}
extra = set(clean_manifest["rule_counts"]) - set(exp["rule_counts"])
print("rows pinned:", sum(rows_ok.values()), "/", len(rows_ok), "| rule counters pinned:", sum(rules_ok.values()), "/", len(rules_ok), "| unpinned counters:", extra or "none")
assert all(rows_ok.values()) and all(rules_ok.values()) and not extra

rows pinned: 9 / 9 | rule counters pinned: 19 / 19 | unpinned counters: none


## 5 · Cross-source consistency (*reported, never repaired* — assumption G3)
Where two sources describe the same thing, do they agree? Numbers are pinned; they are findings about the **sources**, not bugs.

In [7]:
cross = V.cross_source_report(tables, raw)
cmp = pd.DataFrame({"measured": cross, "pinned": exp["cross_source"]})
display(cmp)
assert (cmp.measured == cmp.pinned).all(), "a cross-source number changed"

,measured,pinned
kev_cves_with_pkev_mapping,419,419
kev_cves_without_pkev_mapping,972,972
kev_and_nvd_both_have_informative_cwe,1148,1148
kev_nvd_cwe_sets_equal,863,863
kev_nvd_cwe_sets_disjoint,126,126
kev_added_before_nvd_published,55,55
nvd_missing_kev_echo_fields,3,3
nvd_kev_echo_value_differences,0,0
pkev_technique_name_differs_from_attack,0,0
pkev_cve_description_differs_from_kev_name,3,3


**How to read them (for the report / exam):**
* **972 of 1,391 KEV CVEs have no PKEV mapping.** "Not mapped" ≠ "no technique applies" (P5). Coverage must be shown in every result.
* **KEV vs NVD CWE:** among 1,148 CVEs where both give an informative CWE, only 863 agree exactly and 126 share nothing. Two sources, two opinions: both are stored, neither wins (G3).
* **55 CVEs entered KEV before NVD's `published` date** (different clocks; do not assume NVD is always first).
* **NVD echo of KEV fields** (`cisaExploitAdd`…) is missing for 3 recent CVEs (NVD mirror synced before KEV's last update) and has **0** value differences elsewhere: good evidence the two snapshots are aligned.
* **PKEV vs ATT&CK names:** 0 differences. **PKEV vs KEV names:** 3 rows (one CVE, KEV renamed it).

## 6 · Coverage preview (what Phase 3/4 will be able to answer)

In [8]:
kev, nvd, pk, att = tables["kev_vulnerability"], tables["nvd_cve"], tables["pkev_mapping"], tables["attack_technique"]
cov = pd.Series({
    "KEV CVEs (scope)": len(kev),
    "… with an NVD record": int(kev.cve_id.isin(nvd.cve_id).sum()),
    "… with a CVSS v3.1 score": int(nvd.cvss_v31_score.notna().sum()),
    "… with an informative CWE (NVD)": int(tables["nvd_cve_cwe"].query("is_informative").cve_id.nunique()),
    "… with ≥1 PKEV technique mapping": int(pk.cve_id.nunique()),
    "ATT&CK techniques (all statuses)": len(att),
    "… active": int((att.status == "active").sum()),
    "… used by PKEV": int(pk.technique_id.nunique()),
}, name="count")
display(cov.to_frame())

,count
KEV CVEs (scope),1391
… with an NVD record,1391
… with a CVSS v3.1 score,1389
… with an informative CWE (NVD),1188
… with ≥1 PKEV technique mapping,419
ATT&CK techniques (all statuses),799
… active,656
… used by PKEV,155


## 7 · Negative tests — do the checks really fail on broken tables?
We damage **copies** (in memory, or temp files) in 14 different ways. Each must be detected by the right family.

In [9]:
def broken(table, fn):
    t = dict(tables); df = tables[table].copy(); fn(df); t[table] = df; return t
detected = lambda rs: any(not r.ok for r in rs)
T = []   # (what we broke, family expected to catch it, caught?)

T.append(("duplicate primary key (kev_vulnerability)", "keys", detected(V.check_keys(broken("kev_vulnerability", lambda d: d.loc.__setitem__((1, "cve_id"), d.loc[0, "cve_id"]))))))
T.append(("orphan foreign key (pkev_mapping.cve_id)", "keys", detected(V.check_keys(broken("pkev_mapping", lambda d: d.loc.__setitem__((0, "cve_id"), "CVE-1999-0001"))))))
T.append(("orphan technique (pkev_mapping.technique_id)", "keys", detected(V.check_keys(broken("pkev_mapping", lambda d: d.loc.__setitem__((0, "technique_id"), "T9999"))))))
T.append(("NULL in a mandatory column (kev.product)", "schema", detected(V.check_schema(broken("kev_vulnerability", lambda d: d.loc.__setitem__((0, "product"), pd.NA))))))
T.append(("blank string (kev.notes)", "schema", detected(V.check_schema(broken("kev_vulnerability", lambda d: d.loc.__setitem__((0, "notes"), "  "))))))
T.append(("padded string (nvd.description_en)", "schema", detected(V.check_schema(broken("nvd_cve", lambda d: d.loc.__setitem__((0, "description_en"), " x "))))))
T.append(("CVSS score 11 (out of range)", "domain", detected(V.check_domain(broken("nvd_cve", lambda d: d.loc.__setitem__((0, "cvss_v31_score"), 11.0)), SRC))))
T.append(("unknown mapping_type", "domain", detected(V.check_domain(broken("pkev_mapping", lambda d: d.loc.__setitem__((0, "mapping_type"), "guess")), SRC))))
T.append(("KEV date_added in the future", "domain", detected(V.check_domain(broken("kev_vulnerability", lambda d: d.loc.__setitem__((0, "date_added"), pd.Timestamp("2030-01-01"))), SRC))))
T.append(("severity contradicts score (9.8 but LOW)", "semantic", detected(V.check_semantic(broken("nvd_cve", lambda d: d.loc.__setitem__((d.cvss_v31_score.first_valid_index(), "cvss_v31_severity"), "LOW"))))))
T.append(("is_subtechnique flag wrong", "semantic", detected(V.check_semantic(broken("attack_technique", lambda d: d.loc.__setitem__((d.index[d.is_subtechnique][0], "is_subtechnique"), False))))))
T.append(("a row silently lost (nvd_cve)", "reconcile", detected(V.check_reconcile(broken("nvd_cve", lambda d: d.drop(d.index[0], inplace=True)), expected))))

tmp = Path(tempfile.mkdtemp(prefix="tg_neg2_"))
shutil.copytree(CLEAN, tmp / "clean")
f = tmp / "clean" / "kev_vulnerability.csv"; b = bytearray(f.read_bytes()); b[2000] ^= 1; f.write_bytes(bytes(b))
T.append(("one bit flipped in a clean CSV", "provenance", detected(V.check_provenance(tmp / "clean", clean_manifest, RAW, raw_manifest, tables))))
shutil.rmtree(tmp)

# G1: a raw file modified -> cleaning must refuse to start
tmp = Path(tempfile.mkdtemp(prefix="tg_neg3_")); rec = next(r for r in raw_manifest["files"] if r["relpath"].endswith("known_exploited_vulnerabilities.json"))
(tmp / "kev").mkdir(); g = tmp / "kev" / "known_exploited_vulnerabilities.json"; shutil.copyfile(RAW / rec["relpath"], g)
b = bytearray(g.read_bytes()); b[500] ^= 1; g.write_bytes(bytes(b))
try: C.verify_raw_inputs(tmp, {"files": [rec]}); caught = False
except IntegrityError: caught = True
T.append(("raw file modified -> cleaning refuses (G1)", "provenance", caught)); shutil.rmtree(tmp)

neg = pd.DataFrame(T, columns=["what we broke", "family that must catch it", "caught"])
display(neg); assert neg.caught.all(), "a broken input slipped through"
print("all", len(neg), "negative tests detected")

,what we broke,family that must catch it,caught
0,duplicate primary key (kev_vulnerability),keys,True
1,orphan foreign key (pkev_mapping.cve_id),keys,True
2,orphan technique (pkev_mapping.technique_id),keys,True
3,NULL in a mandatory column (kev.product),schema,True
4,blank string (kev.notes),schema,True
5,padded string (nvd.description_en),schema,True
6,CVSS score 11 (out of range),domain,True
7,unknown mapping_type,domain,True
8,KEV date_added in the future,domain,True
9,severity contradicts score (9.8 but LOW),semantic,True


all 14 negative tests detected


## 8 · Phase gate

In [10]:
V.assert_all(results)
report = V.write_report(CLEAN / "phase2_validation_report.json", results,
    {"clean_manifest_sha256": sha256_file(CLEAN / "manifest.json"), "raw_manifest_sha256": sha256_file(config.MANIFEST_PATH),
     "sources_digest_sha256": config.sources_digest(), "negative_tests_detected": int(neg.caught.sum()), "negative_tests_total": len(neg),
     "cross_source": cross})
print("PHASE 2 GATE: PASSED ->", report["summary"])

PHASE 2 GATE: PASSED -> {'schema': {'passed': 45, 'failed': 0}, 'keys': {'passed': 22, 'failed': 0}, 'domain': {'passed': 28, 'failed': 0}, 'semantic': {'passed': 8, 'failed': 0}, 'reconcile': {'passed': 19, 'failed': 0}, 'provenance': {'passed': 30, 'failed': 0}}


## 9 · What this does **not** prove
* Clean = *consistent and traceable*, not *true*. KEV/NVD/PKEV can still be wrong at the source.
* NVD timestamps are **assumed** to be UTC (G5) [UNCERTAIN]. CVSS: only v3.1 is modelled; v4.0 and CPE configurations are ignored (N5).
* The independent recount uses the same author's understanding of the rules (two code paths, one mind): it catches coding errors, not a wrong *decision*. Decisions are reviewed in `docs/phase2_cleaning_assumptions.md`.
* The vulnStatus list is a superset of the 4 values observed [UNCERTAIN completeness].

**Next (Phase 3, concept only):** UML → Django models for these 10 tables, then load `data/clean/` and compare DB counts with this notebook.